# IBVAP — GPU Video Analytics Pipeline (Colab)

Drop videos into `input_videos/`, run all cells, collect annotated videos + full telemetry from `runs/`.

**Stage order:** motion gate → pHash → letterbox → YOLO-seg + ByteTrack → ANPR → FRS → drone → tamper → fence → SGM depth → annotate

## 1. Check the GPU
Colab free tier gives a T4. Confirm it before running.

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Drop your videos here
Run this cell, then use the **Choose Files** button. Any of mp4/avi/mov/mkv/webm/ts work.
Name each file after its camera ID (e.g. `c01.mp4`) if you want per-camera calibration.

In [ ]:
import os
from google.colab import files
os.makedirs('input_videos', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    os.replace(name, os.path.join('input_videos', name))
print('videos:', sorted(os.listdir('input_videos')))

## 3. Install dependencies
Takes ~2 minutes first time. `pynvml` + `psutil` drive the telemetry sampler.

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy pynvml psutil
import ultralytics, cv2, numpy
print('ultralytics', ultralytics.__version__, '| opencv', cv2.__version__)

## 4. Add the pipeline source
Option A — clone/copy this repo's `ibvap/` folder and `run_ibvap.py`.
Option B — mount Drive if you keep the code there.

In [ ]:
# Option A: upload the ibvap package + run_ibvap.py
from google.colab import files
import zipfile, os
up = files.upload()
for name, _ in up.items():
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
        os.remove(name)
print('ibvap present:', os.path.isdir('ibvap'), '| runner present:', os.path.isfile('run_ibvap.py'))

# Option B: mount Drive instead
# from google.colab import drive
# drive.mount('/content/drive')

## 5. (Optional) camera calibration + fence zones
Calibration drives SGM depth. Fence zones are pure geometry — no model.
Keys are video filename stems.

In [ ]:
import json, os
calib = {
  "c01": {"mode": "ground", "horizon_y": 42, "cam_height_m": 6.0, "focal_px": 1180, "rig_m": 1.2},
  "c02": {"mode": "ground", "horizon_y": 45, "cam_height_m": 5.0, "focal_px": 940,  "rig_m": 0.8},
  "c03": {"mode": "aerial", "fixed_depth_m": 120},
  "c04": {"mode": "ground", "horizon_y": 48, "cam_height_m": 4.0, "focal_px": 1620, "rig_m": 0.6},
  "c11": {"mode": "sky"}
}
json.dump({"camera_calib": calib}, open('calib.json', 'w'), indent=2)

zones = {
  "c01": [{"id": "VF-02", "name": "VF-02 Fence North", "kind": "tripwire",
           "points": [[0.08, 0.72], [0.94, 0.55]], "direction": "both", "min_size_pct": 3}],
  "c06": [{"id": "VF-04", "name": "VF-04 Treeline", "kind": "polygon",
           "points": [[0.16, 0.40], [0.58, 0.32], [0.86, 0.58], [0.40, 0.74]], "min_size_pct": 3}]
}
json.dump(zones, open('zones.json', 'w'), indent=2)
print('calib.json + zones.json written')

## 6. Run the pipeline
Everything is timed and written to a timestamped folder under `runs/`.

In [ ]:
!python run_ibvap.py \
  --input input_videos \
  --output runs \
  --config calib.json \
  --zones zones.json \
  --device 0 \
  --imgsz 640 \
  --conf 0.25 \
  --run-name colab-t4

## 7. Read the telemetry
`summary.json` is the aggregated report. `gpu_timeline.csv` and `stage_latency.csv` are per-frame.

In [ ]:
import json, glob, os
run = sorted(glob.glob('runs/*'))[-1]
print('run dir:', run)
s = json.load(open(os.path.join(run, 'summary.json')))
print(json.dumps({
  'wall_clock': s['wall_clock'],
  'frames': s['frames'],
  'throughput': s['throughput'],
  'gpu_util': s['gpu']['utilisation_pct'],
  'errors': s['errors'],
}, indent=2))
print('\nstage share:')
for k, v in sorted(s['stage_share_pct'].items(), key=lambda x: -x[1]):
    print(f"  {k:24s} {v:6.2f}%  p95={s['stages'][k]['p95_ms']:8.2f}ms  calls={s['stages'][k]['calls']}")
print('\nfiles:')
for f in sorted(os.listdir(run)):
    print('  ', f, os.path.getsize(os.path.join(run, f)) if os.path.isfile(os.path.join(run, f)) else '<dir>')

## 8. Download results

In [ ]:
import glob, os
run = sorted(glob.glob('runs/*'))[-1]
for f in glob.glob(os.path.join(run, 'videos', '*.mp4')):
    files.download(f)
files.download(os.path.join(run, 'summary.json'))
files.download(os.path.join(run, 'stage_latency.csv'))
files.download(os.path.join(run, 'gpu_timeline.csv'))
files.download(os.path.join(run, 'events.csv'))

## Notes

- **Reconnect runtime before each run** to get a fresh GPU quota.
- **`--max-frames 300`** for a quick smoke test before a full run.
- **`--no-video`** if you only want telemetry (much faster).
- **`--half`** enables FP16; leave off if you hit numerical issues.
- Colab disconnects after ~90 min idle — long batches should be chunked.
- Output videos use `avc1` codec; the writer falls back to `mp4v` automatically.